In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
DIANNE = ".."
sys.path.insert(0, f"{DIANNE}/dianne-core")
sys.path.insert(0, f"{DIANNE}/dianne-utils")
sys.path.insert(0, f"{DIANNE}/dianne-viewer")
import dianne_core
import dianne_utils as dianne
import dianne_viewer as viewer

import pandas as pd
import matplotlib.colors as mcolors
dianne.setNotebookWidth(100)

In [ ]:
stqDataPath = '/projects/chuang-lab/PDXnet/PDX_TMA_ST/analysis/image_analysis/phaseII/stq/arbitrary_grid/50_micron_tiles/'
xeniumBundlesDir = '/projects/chuang-lab/PDXnet/PDX_TMA_ST/analysis/image_analysis/phaseII/DIANNE/xenium_bundles'
XTRACT_DIR   = '/projects/chuang-lab/PDXnet/PDX_TMA_ST/analysis/xenium_phaseII/XTRACT'
REGISTRY_CSV = f'{XTRACT_DIR}/reports/tissue_data_registry.csv'

samples = [s for s in sorted(os.listdir(stqDataPath)) if not 'pipeline_info' in s][:]

idm = './identity-matrix.csv'
xenium_to_he_matrices = {sample: idm for sample in samples}

species = {}
mouse_cell_type = {}
mouse_cell_type_broad = {}
xenium_bundle_paths = {}
for s in samples:
    xenium_bundle_paths.update({s: None})
    if '-CORE' in s:
        sid = s.split("-CORE")[0]
        xenium_bundle_paths.update({s: f'{xeniumBundlesDir}/{sid}/'})
        species.update({s: pd.Categorical(pd.read_csv(f'{xeniumBundlesDir}/{sid}/species.csv', header=None)[0])})
        mouse_cell_type.update({s: pd.Categorical(pd.read_csv(f'{xeniumBundlesDir}/{sid}/mouse_cell_type.csv', header=None)[0])})
        mouse_cell_type_broad.update({s: pd.Categorical(pd.read_csv(f'{xeniumBundlesDir}/{sid}/mouse_cell_type_broad.csv', header=None)[0])})
all_annotations = [species, mouse_cell_type, mouse_cell_type_broad]
annotationsPalette = [{a: mcolors.to_hex(dianne.Set123(i)) for i, a in enumerate(labels[samples[0]].categories)} for labels in all_annotations]

se_morph = XTRACT_DIR + '/' + pd.read_csv(REGISTRY_CSV).set_index('sample_id')['Spatial Data Path'].str.replace('_sdata.zarr', '_xenium_explorer/morphology.ome.tif')
secondary_images = {}
for s in samples:
    if '-CORE' in s:
        found = se_morph[s.split('-CORE')[0]]
        if len(found)>1:
            found = found.iloc[0]
        secondary_images.update({s: found})
    else:
        secondary_images.update({s: f'{stqDataPath}/{s}/image.ome.tiff'})

secondary_matrices = xenium_to_he_matrices.copy()

classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

load_features = False

if load_features:
    F = 1
    patch_size = 6 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
    ts, mpp, tile_size = dianne.loadSTQParams(stqDataPath + samples[0], F)
    fname = f'img.data.uni2-{F}.h5ad'
    ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne_core.loadDataAndPreparePatches(samples, stqDataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
    sizes = {s: ads[s].shape[0] for s in samples}
    print(f'Prepared {patchesCDFs.shape[0]} patches')
    
    runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2)
    savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
    loadfn = dianne.makeLoadFn(classifierPaths)
    listfn = dianne.makeListFn(classifierPaths)
else:
    runfn, savefn, loadfn, listfn, sizes = None, None, None, None, None
    imgs = {s: f'{stqDataPath}/{s}/image.ome.tiff' for s in samples}

drawings = viewer.create_viewer(samples, secondary_images, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                                xenium_mpp=0.2125, max_cells=20000, matrices=xenium_to_he_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                secondary_images=imgs, secondary_matrices=secondary_matrices, draw_on_secondary=True,
                                annotations=all_annotations, category_colors=annotationsPalette,
                                save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]